# Step 11 — the same fit, for cohorts B and C

Step 02 fits cohort A. Steps 12–16 compare all three, so B and C need the **identical** fit:
same power, same `minModuleSize`, same merge height, same seed. Anything else and a difference
between cohorts could be a difference in parameters rather than in the data.

Reads `cohorts/`. Writes `data/run_artifacts/wgcna_B.rds` and `wgcna_C.rds`, in exactly the shape
step 02 writes `wgcna_A.rds`, so every later notebook can treat the three alike.

**This is the slow step: two full `blockwiseModules` runs, roughly 80 seconds each.**

Nothing here is tuned per cohort. The parameters come from steps 01–02 and are repeated verbatim:
the power floor of 12 is the WGCNA authors' published floor for a signed network at n > 40, and
`minModuleSize = 5` is where the interferon signature resolves without the panel shattering
(step 02 has the scan).

In [ ]:
suppressMessages(library(WGCNA))
source("../src/paths.R")
options(stringsAsFactors = FALSE); enableWGCNAThreads(4)

POWER <- 12; MIN_MODULE <- 5; MERGE_CUT <- 0.25; SEED <- 42   # identical to step 02

for (COHORT in c("B", "C")) {
  set.seed(SEED)
  X <- read.csv(coh("R_cohort-%s_log2_combat.csv", COHORT), row.names = 1, check.names = FALSE)
  m <- read.csv(coh("R_cohort-%s_meta.csv",        COHORT), row.names = 1, check.names = FALSE)
  m <- m[rownames(X), , drop = FALSE]

  net  <- blockwiseModules(X, power = POWER, networkType = "signed",
                           minModuleSize = MIN_MODULE, mergeCutHeight = MERGE_CUT,
                           numericLabels = TRUE, maxBlockSize = 8000, verbose = 0)
  mods <- labels2colors(net$colors)
  ME   <- moduleEigengenes(X, mods)$eigengenes

  # Locate the interferon module the same way step 02 does: by overlap with the
  # curated ISG set, never by colour. It is allowed to fail -- if no module holds
  # three of the set, this cohort has no interferon module and says so.
  ifn <- tryCatch(as.character(locate_ifn_module(mods, colnames(X))),
                  error = function(e) NA_character_)

  saveRDS(list(cohort = COHORT, X = as.matrix(X), meta = m, power = POWER,
               mods = mods, ME = ME, ifn = ifn, minModuleSize = MIN_MODULE,
               mergeCutHeight = MERGE_CUT, seed = SEED),
          art("wgcna_%s.rds", COHORT))
  write.csv(data.frame(protein = colnames(X), module = mods),
            art("modules_%s.csv", COHORT), row.names = FALSE)

  cat(sprintf("cohort %s: n = %d, %d modules (excl. grey), %d probes unassigned, interferon = %s\n",
              COHORT, nrow(X), length(setdiff(unique(mods), "grey")),
              sum(mods == "grey"), ifn))
}

## The three fits are not alike, and that is a result

Same study, same parameters, a random split of the same 260 donors:

| cohort | n | modules | unassigned | largest module |
|---|---|---|---|---|
| A | 87 | 52 | 1,016 | 2,216 (30.4%) |
| B | 87 | 23 | 1,667 | 2,784 (38.2%) |
| C | 86 | 46 | 1,377 | 2,656 (36.4%) |

B's network resolves into **less than half** as many modules as A's and leaves 64% more protein
unassigned. That is not a parameter choice — the parameters are identical — and step 16 shows it is
not the patients either, since batch composition is fixed by construction and trait spread is close
across all three.

It is the reason cohort B produces one trait-associated module where A produces twelve, and it is
the single most important caveat on everything in steps 12–16: **module discovery at n ≈ 87 is not
stable across a random split of the same cohort.**